## Strategy Feature Reference

Each trading day × battery combination is described by 22 features before being passed to UMAP + clustering. All features are z-score normalised before dimensionality reduction.

---

### Revenue composition

| Feature | Description | Formula |
|---|---|---|
| `fcas_revenue_share` | Fraction of total revenue from FCAS markets | $\frac{\sum \text{FCAS}}{\sum |\text{energy net}| + \sum \text{FCAS}}$ |
| `reg_vs_contingency_ratio` | How FCAS revenue splits between regulation (slow, continuous) and contingency (fast, event-based) | $\frac{\sum(\text{raise\_reg} + \text{lower\_reg})}{\sum \text{FCAS}}$ |
| `raise_vs_lower_bias` | Whether the battery earns more from raise or lower FCAS services | $\frac{\sum(\text{raise\_contingency} + \text{raise\_reg})}{\sum \text{FCAS}}$ |
| `revenue_diversity_index` | Shannon entropy across 5 revenue streams (energy, raise_reg, lower_reg, raise_contingency, lower_contingency); higher = more diversified | $-\sum_i p_i \ln p_i$ |
| `co_optimization_frequency` | Fraction of intervals where the battery earned both energy and FCAS revenue simultaneously | $\overline{\mathbb{1}[|\text{energy\_mw}| > 0 \;\wedge\; \text{FCAS} > 0]}$ |
| `revenue_per_mw` | Daily net revenue normalised by peak discharge MW; higher = earning more per unit of capacity dispatched | $\frac{\sum \text{net}}{\max(\text{discharge\_mw})}$ |

---

### Dispatch behaviour

| Feature | Description | Formula |
|---|---|---|
| `daily_cycle_count` | Equivalent full cycles per day (energy throughput relative to MWh capacity) | $\frac{\sum \text{discharge\_mw} \times \frac{5}{60}}{\text{MWh\_capacity}}$ |
| `utilization_factor` | Fraction of intervals where the battery was actively charging or discharging | $\overline{\mathbb{1}[\text{discharge} > 0 \;\vee\; \text{charge} > 0]}$ |
| `charge_discharge_ratio` | Ratio of total discharge to total charge MW; close to 1 = balanced, >1 = net exporter | $\frac{\sum \text{discharge\_mw}}{\sum \text{charge\_mw} + \varepsilon}$ |
| `discharge_peak_intensity` | Ratio of peak dispatch to average dispatch when discharging; high = peaky/spike-oriented strategy | $\frac{\max(\text{discharge\_mw})}{\overline{\text{discharge\_mw} \;|\; \text{discharge\_mw} > 0}}$ |
| `resting_time_avg` | Average number of idle intervals between a discharge event and the next charge event | counted from state array $s_t \in \{-1, 0, +1\}$ where $+1=\text{charge},\; 0=\text{idle},\; -1=\text{discharge}$ |

---

### Price behaviour

| Feature | Description | Formula |
|---|---|---|
| `energy_price_correlation` | Pearson correlation between discharge MW and the 5-min RRP; positive = discharging when prices are high | $\text{corr}(\text{discharge\_mw},\; \text{rrp})$ |
| `negative_price_capture` | Average charge MW during negative-price intervals; high = battery absorbs surplus generation | $\overline{\text{charge\_mw} \;|\; \text{rrp} < 0}$ |
| `discharge_strike_price` | 10th-percentile RRP at which the battery discharges; proxy for the lowest price the battery will tolerate | $P_{10}(\text{rrp} \;|\; \text{discharge} > 0)$ |
| `charge_strike_price` | 90th-percentile RRP at which the battery charges; proxy for the highest price the battery will tolerate when buying | $P_{90}(\text{rrp} \;|\; \text{charge} > 0)$ |
| `discharge_price_premium` | How much higher the discharge-weighted RRP is relative to the day's average RRP; >1 = strategic timing above average | $\frac{\sum \text{discharge\_mw} \cdot \text{rrp}}{\sum \text{discharge\_mw}} \bigg/ \overline{\text{rrp}}$ |
| `volatility_capture_rate` | Share of positive net revenue earned during the top 5% most volatile price intervals; high = the battery profits from price spikes | $\frac{\sum_{t \in V_{0.05}} \text{net}^+_t}{\sum_t \text{net}^+_t}$ where $V_{0.05}$ = top 5% by $|\Delta \text{rrp}_t|$ |
| `price_response_speed` | Magnitude-weighted speed of response to significant price moves; for each top-quartile price event, the lag to first matching dispatch (discharge after spike, charge after crash) is penalised with $\frac{1}{1+\ell}$. Score of 1 = always instant, ~0.2 = always misses within 3 intervals | $\frac{\displaystyle\sum_{t \in S} |\Delta\text{rrp}_t| \cdot \frac{1}{1+\ell_t}}{\displaystyle\sum_{t \in S} |\Delta\text{rrp}_t|}$ where $S = \{t : |\Delta\text{rrp}_t| > P_{75}\}$, $\ell_t \in \{0,1,2,3,4\}$ |

---

### Time-of-day patterns

All four features are computed as fractions of total daily discharge (or charge). Values sum to ≤ 1 since only the named window is counted.

| Feature | Window (AEST) | Direction | Formula |
|---|---|---|---|
| `evening_peak_weight` | 17:00 – 21:00 | Discharge | $\frac{\sum_{t \in [17,21)} \text{discharge\_mw}_t}{\sum_t \text{discharge\_mw}_t}$ |
| `morning_peak_weight` | 06:00 – 09:00 | Discharge | $\frac{\sum_{t \in [6,9)} \text{discharge\_mw}_t}{\sum_t \text{discharge\_mw}_t}$ |
| `solar_soak_charge_weight` | 10:00 – 15:00 | Charge | $\frac{\sum_{t \in [10,15)} \text{charge\_mw}_t}{\sum_t \text{charge\_mw}_t}$ |
| `overnight_charge_weight` | 00:00 – 04:00 | Charge | $\frac{\sum_{t \in [0,4)} \text{charge\_mw}_t}{\sum_t \text{charge\_mw}_t}$ |


In [ ]:
from nem_battery.strategy_map import (
    load_interval_data,
    prepare_interval_data,
    build_feature_frame,
    train_strategy_model,
    FEATURE_COLUMNS,
    EPSILON,
    plot_cluster_battery_count,
    StrategyArtifacts,
    plot_cluster_feature_means,
    plot_cluster_revenue,
)

from nem_battery.battery import KNOWN_BATTERIES
import plotly.express as px
import duckdb

mwh_capacity_map = {v.name: v.mwh_capacity for k, v in KNOWN_BATTERIES.items()}
mwh_capacity_map

In [ ]:
# generate the features dataframe
raw = load_interval_data(target="local")
# raw = raw[raw["battery_key"] != "lake_bonney"]
prepared = prepare_interval_data(raw)
features = build_feature_frame(prepared)

n_clusters = 4
artifacts = train_strategy_model(
    features,
    clusterer="kmeans",
    n_clusters=n_clusters,
)

In [ ]:
plot_cluster_battery_count(artifacts)

In [ ]:
plot_cluster_feature_means(artifacts)

In [ ]:
def plot_cluster_revenue(artifacts: StrategyArtifacts) -> None:
    embedding_df = artifacts.embedding_3d
    mwh_capacity_map = {v.name: v.mwh_capacity for k, v in KNOWN_BATTERIES.items()}

    embedding_df["net_revenue_per_mwh"] = embedding_df.apply(
        lambda row: row["net_revenue"] / mwh_capacity_map.get(row["battery_name"], 1), axis=1
    )
    cluster_revenue = embedding_df.groupby("cluster")[["net_revenue", "net_revenue_per_mwh"]].agg(
        ["min", "mean", "max"]
    )

    cluster_revenue.columns = ["_".join(col) for col in cluster_revenue.columns]
    cluster_revenue = cluster_revenue.reset_index().set_index("cluster")
    return cluster_revenue
    # cluster_revenue = cluster_revenue.T
    # fig = px.imshow(
    #     cluster_revenue,
    #     labels={"x": "Cluster", "y": "Feature", "color": "Relative Value"},
    #     color_continuous_scale="Blues",
    #     aspect="auto",
    # )

    # fig.update_layout(
    #     title="Cluster Feature Means (Relative)",
    #     xaxis_tickangle=0,
    # )

    # fig.show()


plot_cluster_revenue(artifacts)